In [1]:
import os
import subprocess
import sys

import pandas as pd
import numpy as np

print("Start data generation...")
os.chdir('/home/jovyan')

result = subprocess.run(['python', 'scripts/generate_sample_data.py'], capture_output = True, text=True)
print(result.stdout)

Start data generation...
Generate Date Dimension
Generating customer dimension...
Generated 50000 customer records
Generating product dimension...
Generated 5000 product records
Generating transaction fact table...
  Generated 100000 transactions...
  Generated 200000 transactions...
  Generated 300000 transactions...
  Generated 400000 transactions...
  Generated 500000 transactions...
  Generated 600000 transactions...
  Generated 700000 transactions...
  Generated 800000 transactions...
  Generated 900000 transactions...
  Generated 1000000 transactions...
Saved chunk 1
Saved chunk 2
Saved chunk 3
Saved chunk 4
Saved chunk 5
Saved chunk 6
Saved chunk 7
Saved chunk 8
Saved chunk 9
Saved chunk 10
Generated 1000000 transaction records
Generating streaming data samples...
Generated 100000 streaming transaction records

=== Data Generation Complete ===
Customers: 50,000
Products: 5,000
Dates: 731
Transactions: 1,000,000

Files saved in data/raw/



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("GlobalMart-ECommerce-Analytics") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

In [4]:
customers_df = spark.read \
    .option("header", "true") \
    .csv("/home/jovyan/data/raw/dim_customers.csv")

In [5]:
customers_df.createOrReplaceTempView("dim_customers")

In [6]:
spark.sql(""" 
    SELECT * FROM dim_customers
""").show(1)

+-----------+----------+---------+-----------------+-----------------+----------------+---------+---------+---------+
|customer_id|first_name|last_name|            email|registration_date|customer_segment|  country|     city|age_group|
+-----------+----------+---------+-----------------+-----------------+----------------+---------+---------+---------+
|          1|   William|   Miller|user1@example.com|       2023-08-30|         regular|Singapore|Singapore|    46-55|
+-----------+----------+---------+-----------------+-----------------+----------------+---------+---------+---------+
only showing top 1 row



In [7]:
date_df = spark.read \
    .option("header", "true") \
    .csv("/home/jovyan/data/raw/dim_date.csv")
products_df = spark.read \
    .option("header", "true") \
    .csv("/home/jovyan/data/raw/dim_products.csv")

In [8]:
transactions_df = spark.read \
    .option("header", "true") \
    .csv("/home/jovyan/data/raw/fact_transactions*.csv")

In [9]:
date_df.createOrReplaceTempView("dim_date")
products_df.createOrReplaceTempView("dim_products")
transactions_df.createOrReplaceTempView("fact_transactions")

In [11]:
spark.sql(""" 
    SELECT * FROM fact_transactions
""").show(1, vertical=True)

-RECORD 0------------------------------------
 transaction_id        | 300001              
 customer_id           | 5961                
 product_id            | 4414                
 date_id               | 20241111            
 region_id             | 1                   
 transaction_timestamp | 2024-11-11 20:10:47 
 quantity              | 5                   
 unit_price            | 1127.61             
 total_amount          | 5638.05             
 discount_amount       | 0.0                 
 tax_amount            | 563.81              
 payment_method        | Credit Card         
 transaction_status    | completed           
 fraud_score           | 0.1878              
only showing top 1 row



In [12]:
spark.sql(""" 
    SELECT c.customer_id as customer_id, sum(f.total_amount) as clv 
    FROM fact_transactions f JOIN dim_customers c ON f.customer_id = c.customer_id
    WHERE f.transaction_status = 'completed'
    GROUP BY c.customer_id
    ORDER BY clv DESC
    LIMIT 10
""").show(100, vertical=True)

-RECORD 0------------------------
 customer_id | 47607             
 clv         | 80542.18          
-RECORD 1------------------------
 customer_id | 16339             
 clv         | 78552.11          
-RECORD 2------------------------
 customer_id | 10745             
 clv         | 75767.39          
-RECORD 3------------------------
 customer_id | 1126              
 clv         | 75230.01000000001 
-RECORD 4------------------------
 customer_id | 48018             
 clv         | 75134.56999999999 
-RECORD 5------------------------
 customer_id | 24494             
 clv         | 74410.06          
-RECORD 6------------------------
 customer_id | 41433             
 clv         | 74072.94          
-RECORD 7------------------------
 customer_id | 45834             
 clv         | 73418.28          
-RECORD 8------------------------
 customer_id | 7753              
 clv         | 73096.94          
-RECORD 9------------------------
 customer_id | 33107             
 clv         |

In [13]:
spark.sql(""" 
    SELECT * FROM dim_customers
    WHERE customer_id = 47607
""").show(1)

+-----------+----------+---------+--------------------+-----------------+----------------+-------+------+---------+
|customer_id|first_name|last_name|               email|registration_date|customer_segment|country|  city|age_group|
+-----------+----------+---------+--------------------+-----------------+----------------+-------+------+---------+
|      47607|      John|    Jones|user47607@example...|       2023-06-16|         regular|    USA|London|    46-55|
+-----------+----------+---------+--------------------+-----------------+----------------+-------+------+---------+

